# 06 · NLP, Recurrent Models, BERT, and GPT

Move from TF-IDF and local tokenization through LSTM/GRU baselines, self-attention, masked-language pretraining, classification fine-tuning, causal generation, perplexity, and honest synthetic limitations.

This is a learning and interview notebook, not a thin launcher. It combines mechanism,
executable evidence, diagnostics, limitations, and experiments. Run from top to bottom;
keep `RUN_TRAINING=False` to inspect committed/CI results, or turn it on to reproduce them.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Learning objectives

- Establish TF-IDF baselines before neural models.
- Inspect deterministic local tokenization and padding masks.
- Compare LSTM/GRU, bidirectional BERT-style attention, and causal GPT-style attention.
- Distinguish pretraining objective from downstream fine-tuning.
- Interpret perplexity and generated samples without claiming general language ability.

In [ ]:
if not (DATA_DIR / "manifest.json").exists():
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--profile", "ci", "--output", str(DATA_DIR)],
        check=True,
    )
subprocess.run([sys.executable, "scripts/validate_data.py", "--data-dir", str(DATA_DIR)], check=True)
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(manifest["dataset_version"], manifest["rows"])

In [ ]:
import pandas as pd
texts = pd.read_csv(DATA_DIR / "support_texts.csv.gz")
display(texts.sample(10, random_state=20260820))
display(pd.crosstab(texts.intent, texts.sentiment, normalize="index"))

## TF-IDF baseline

TF-IDF increases weight for terms frequent in a document but uncommon in the corpus. Logistic
regression then learns a linear boundary. On templated synthetic text this can be extremely strong;
that does not imply semantic understanding.

In [ ]:
metric_path = METRICS_DIR / "nlp.json"
if RUN_TRAINING or not metric_path.exists():
    metric_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "scripts/run_nlp.py", "--data-dir", str(DATA_DIR), "--profile", "ci", "--output", str(METRICS_DIR / "nlp.json")], check=True)
result = json.loads(metric_path.read_text())

print("loaded", metric_path, "dataset", result.get("_run", {}).get("dataset_version"))

In [ ]:
print("TF-IDF", result["tfidf"])

## Local tokenizer

The vocabulary is built only from generated text. `[CLS]` starts a sequence, `[SEP]` ends it,
`[PAD]` fills fixed length, `[UNK]` handles unseen words, and `[MASK]` supports MLM.

In [ ]:
from rookie_lab.nlp.tokenizer import LocalTokenizer
tokenizer = LocalTokenizer(texts.text.tolist())
ids, mask = tokenizer.batch(texts.text.head(3).tolist())
print("vocabulary", len(tokenizer.tokens))
display(pd.DataFrame(ids.numpy()))
display(pd.DataFrame(mask.numpy()))
print(tokenizer.decode(ids[0].tolist()))

## LSTM and GRU text baselines

Word embeddings feed recurrent cells. LSTM uses a cell state and three gates; GRU is smaller with
update/reset gates. The last hidden state drives sentiment classification.

In [ ]:
recurrent = pd.DataFrame(result["recurrent_baselines"]).T
display(recurrent[["accuracy", "roc_auc", "parameters", "training_loss"]])

## Scaled dot-product self-attention

$Attention(Q,K,V)=softmax(QK^T/\sqrt{d_k})V$. Multi-head attention learns several projections;
padding and causal masks determine which tokens are visible.

In [ ]:
import torch

torch.manual_seed(7)
tokens = torch.randn(1, 4, 8)
scores = tokens @ tokens.transpose(-1, -2) / np.sqrt(tokens.shape[-1])
weights = scores.softmax(-1)
attended = weights @ tokens
print("weights sum", weights.sum(-1))
display(pd.DataFrame(weights[0].numpy()))

## BERT-style encoder

BERT sees left and right context. Here one encoder block with four heads is first optimized on masked
tokens, then copied into a sentiment classifier. The model is random-initialized and tiny.

In [ ]:
bert = result["bert"]
print(bert["architecture"])
print(bert["attention"])
display(pd.DataFrame({"scratch": bert["from_scratch"], "pretrained": bert["pretrained_then_finetuned"]}).T)

## Pretraining versus scratch

One CI MLM update can have zero meaningful loss change and does not guarantee downstream benefit.
The experiment intentionally records that limitation. Portfolio mode uses more updates but remains
far from foundation-model pretraining.

In [ ]:
plt.plot(bert["masked_language_pretraining_loss"], marker="o")
plt.title("Masked-language pretraining loss"); plt.xlabel("update"); plt.show()

## GPT-style decoder and causal mask

GPT predicts the next token and cannot attend to future positions. The upper-triangular causal mask
sets future attention logits to negative infinity before softmax.

In [ ]:
length = 8
causal_mask = torch.triu(torch.ones(length, length), diagonal=1).bool()
display(pd.DataFrame(causal_mask.numpy()).style.background_gradient(cmap="Reds"))

## Perplexity and generation

Perplexity is $exp(cross\ entropy)$. Lower is better on the same tokenization/data, but it does not
measure factuality, helpfulness, or safety. Always inspect outputs.

In [ ]:
gpt = result["gpt"]
print("perplexity", gpt["perplexity"])
display(pd.Series(gpt["greedy_generations"], name="generation").to_frame())
plt.plot(gpt["training_loss"]); plt.title("Causal-LM batch loss"); plt.show()

## NLU versus NLG

Intent/sentiment are NLU-style classification tasks. Causal generation is NLG. A model can score
perfectly on synthetic intent labels while producing repetitive text; those are different abilities.

In [ ]:
summary = pd.DataFrame([
    {"model": "TF-IDF", "sentiment_roc_auc": result["tfidf"]["sentiment_roc_auc"]},
    *({"model": name.upper(), "sentiment_roc_auc": values["roc_auc"]} for name, values in result["recurrent_baselines"].items()),
    {"model": "Tiny BERT scratch", "sentiment_roc_auc": bert["from_scratch"]["roc_auc"]},
    {"model": "Tiny BERT pretrained", "sentiment_roc_auc": bert["pretrained_then_finetuned"]["roc_auc"]},
])
display(summary.sort_values("sentiment_roc_auc", ascending=False))

## Exercises and interview checkpoint

1. Add a validation split to GPT and report validation perplexity.
2. Increase MLM steps and test whether transfer consistently helps across seeds.
3. Show one attention matrix for a sentence and state why it is not a causal explanation.
4. Add character n-gram TF-IDF and compare robustness to misspellings.
5. Explain why perfect synthetic classification is not evidence of general NLP capability.